## General Topics Mergers

In [ ]:
import os
import pandas as pd

data_dir = ".."

# we generated technique categorization of clusters elsewhere first
clusters_with_techniques_path = os.path.join(data_dir, "other_outputs", "weaponization_analysis", "clusters_with_weaponization_techniques.csv")
cluster_topics_path = os.path.join(data_dir, "other_outputs", "bertopic_per_cluster_topic_assignments_sorted.csv")
general_topics_path = os.path.join(data_dir, "other_outputs", "bertopic_general_corpus_assignments_sorted.csv")

clusters_with_techniques = pd.read_csv(clusters_with_techniques_path)
cluster_topics = pd.read_csv(cluster_topics_path)
general_topics = pd.read_csv(general_topics_path)

FileNotFoundError: [Errno 2] No such file or directory: '../other_outputs/weaponization_analysis/clusters_with_weaponization_techniques.csv'

In [3]:
clusters_with_techniques.head()

,cluster,source,original_text,weaponization_technique
0,0,Nagorno-Karabakh_conflict_subsampled,"The revision introduces the term ""Armenian Rev...",Glorification & Vilification
1,0,Armenian genocide_subsampled,"The revision introduces the term ""Dashnaks"" in...",Terminology Biasing
2,0,Armenian_Revolutionary_Federation_subsampled,The removed lines contain several phrases that...,Terminology Biasing
3,0,Armenian_Revolutionary_Federation_subsampled,The revision changes the description of the Ar...,Terminology Biasing
4,0,Armenian_Revolutionary_Federation_subsampled,The added lines include significant historical...,Glorification & Vilification


In [4]:
# merge clusters_with_techniques with general_topics on 'source', 'lemmatized_text', and 'original_text'
# BUT our final goal is to extract rows from general_topics that aren't in the merged dataframe
clusters_general_topics_merged = pd.merge(clusters_with_techniques.drop(columns=['cluster']),
                     general_topics.drop(columns=['lemmatized_text']),
                     on=['source', 'original_text'],
                     how='inner')
clusters_general_topics_merged.head()


,source,original_text,weaponization_technique,topic
0,Nagorno-Karabakh_conflict_subsampled,"The revision introduces the term ""Armenian Rev...",Glorification & Vilification,21
1,Armenian genocide_subsampled,"The revision introduces the term ""Dashnaks"" in...",Terminology Biasing,21
2,Armenian_Revolutionary_Federation_subsampled,The removed lines contain several phrases that...,Terminology Biasing,21
3,Armenian_Revolutionary_Federation_subsampled,The revision changes the description of the Ar...,Terminology Biasing,21
4,Armenian_Revolutionary_Federation_subsampled,The added lines include significant historical...,Glorification & Vilification,21


In [ ]:
# get rows from general_topics that aren't in clusters_general_topics_merged
general_topics_exclusive = general_topics.merge(clusters_general_topics_merged,
                                                on=['source', 'original_text', 'topic'],
                                                how='left',
                                                indicator=True)
general_topics_exclusive = general_topics_exclusive[general_topics_exclusive['_merge'] == 'left_only'].drop(columns=['_merge'])
general_topics_exclusive.head()

,topic,source,lemmatized_text,original_text,weaponization_technique
179,1,Armed_Forces_of_Armenia_subsampled,add line word introduce significant shift narr...,The added lines and words introduce a signific...,NaN
225,2,Nagorno-Karabakh_conflict_subsampled,add line introduce narrative frame action nega...,The added lines introduce a narrative that fra...,NaN
229,2,Second Nagorno-Karabakh War_subsampled,include addition phrase emphasize military act...,The revision includes the addition of phrases ...,NaN
233,2,Nakhchivan (city)_subsampled,removed line contain statement frame situation...,The removed lines contained a statement that f...,NaN
235,2,Second Nagorno-Karabakh War_subsampled,add line introduce narrative frame action azer...,The added lines introduce a narrative that fra...,NaN


In [7]:
# save general_topics_exclusive to csv
general_topics_exclusive.to_csv(os.path.join(data_dir, "other_outputs", "entries_exclusive_to_general_topics.csv"), index=False)

In [12]:
# read general_topics_exclusive_with_technique aka the version AFTER the technique has been assigned

general_topics_exclusive_with_technique = pd.read_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "topics-exclusive_with_weaponization_techniques.csv"))
general_topics_exclusive_with_technique.head()

,topic,source,original_text,weaponization_technique
0,1,Armed_Forces_of_Armenia_subsampled,The added lines and words introduce a signific...,Selective Insertion
1,2,Nagorno-Karabakh_conflict_subsampled,The added lines introduce a narrative that fra...,Terminology Biasing
2,2,Second Nagorno-Karabakh War_subsampled,The revision includes the addition of phrases ...,Glorification & Vilification
3,2,Nakhchivan (city)_subsampled,The removed lines contained a statement that f...,Selective Omission
4,2,Second Nagorno-Karabakh War_subsampled,The added lines introduce a narrative that fra...,Glorification & Vilification


In [8]:
clusters_general_topics_merged = clusters_general_topics_merged[['topic', 'weaponization_technique', 'original_text', 'source']]

general_topics_exclusive_with_technique = general_topics_exclusive_with_technique[['topic', 'weaponization_technique', 'original_text', 'source']]

general_topics_with_technique = pd.concat([clusters_general_topics_merged, general_topics_exclusive_with_technique], ignore_index=True)
general_topics_with_technique.head()

,topic,weaponization_technique,original_text,source
0,11,Selective Omission,"The removal of the phrase ""Unlike the Armenian...",Armenian_Revolutionary_Federation_subsampled
1,11,Terminology Biasing,"The addition of the phrase ""{{seealso|Armenian...",Armenian_Revolutionary_Federation_subsampled
2,11,Selective Omission,The revision removes a critical line that stat...,Armenian_Revolutionary_Federation_subsampled
3,11,Selective Insertion,The added lines emphasize the legitimacy and a...,Armenian_Revolutionary_Federation_subsampled
4,11,Glorification & Vilification,"The revision adds the phrase ""in the Diaspora""...",Armenian_Revolutionary_Federation_subsampled


In [ ]:
# save to csv
general_topics_with_technique.to_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_general_topics_with_weaponization_techniques.csv"), index=False)

In [9]:
# merge clusters_with_techniques with cluster_topics on 'source', 'cluster', 'lemmatized_text', and 'original_text'
# maybe inner join since we want only rows with a valid 'topic' assignment
cluster_topics_with_technique = pd.merge(clusters_with_techniques,
                     cluster_topics.drop(columns=['lemmatized_text']),
                     on=['source', 'cluster', 'original_text'],
                     how='inner')  
cluster_topics_with_technique.head()



,cluster,source,original_text,weaponization_technique,topic
0,1,Armenian genocide_subsampled,The revision adds a caption to the image of th...,Terminology Biasing,1
1,1,Armenian genocide_subsampled,The revision removed a significant introductor...,Selective Omission,2
2,1,Armenian genocide_subsampled,The revision introduces a new reference that i...,Glorification & Vilification,2
3,1,Armenian genocide_subsampled,The revision removes a significant quote from ...,Selective Omission,0
4,1,Armenian genocide_subsampled,"The revision changes the section title from ""O...",Terminology Biasing,14


In [ ]:
# save cluster_topics_with_technique to csv
# new column order: cluster,topic,weaponization_technique,original_text,source
cluster_topics_with_technique = cluster_topics_with_technique[['cluster', 'topic', 'weaponization_technique', 'original_text', 'source']]
# sort by cluster and topic
cluster_topics_with_technique = cluster_topics_with_technique.sort_values(by=['cluster', 'topic'])
# save to csv
cluster_topics_with_technique.to_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_cluster_topics_with_weaponization_techniques.csv"), index=False)

In [11]:
cluster_topics.head()

,cluster,topic,source,lemmatized_text,original_text
0,1,0,Armenia_subsampled,add text introduce narrative question recognit...,The added text introduces a narrative that que...
1,1,0,Armenia_subsampled,introduce word place interestingly subtly shif...,"The revision introduces the word ""although"" in..."
2,1,0,Armenia_subsampled,retain significant portion original text genoc...,The revision retains a significant portion of ...
3,1,0,Armenia_subsampled,remove critical phrase label article biased mi...,The revision removed a critical phrase that la...
4,1,0,Armenia_subsampled,introduce subtle shift narrative surround geno...,The revision introduces a subtle shift in the ...


In [15]:
general_topics.head()

,topic,source,lemmatized_text,original_text
0,0,Armenian genocide_subsampled,introduce pov statement point view statement q...,"The revision introduces a ""POV-statement"" (poi..."
1,0,List_of_visitors_to_Tsitsernakaberd_subsampled,add line turkish state official visit tsitsern...,"The added line ""No Turkish state official has ..."
2,0,List_of_visitors_to_Tsitsernakaberd_subsampled,removal line turkish state official visit tsit...,"The removal of the line ""No Turkish state offi..."
3,0,Armenian genocide_subsampled,include addition npov neutral point view wease...,"The revision includes the addition of ""{{NPOV}..."
4,0,Armenian genocide_subsampled,include addition section title websites suppor...,The revision includes the addition of a sectio...


In [16]:
clusters_with_techniques

,cluster,source,original_text,weaponization_technique
0,0,Nagorno-Karabakh_conflict_subsampled,"The revision introduces the term ""Armenian Rev...",Terminology Manipulation
1,0,Armenian genocide_subsampled,"The revision introduces the term ""Dashnaks"" in...",Terminology Manipulation
2,0,Armenian_Revolutionary_Federation_subsampled,The removed lines contain several phrases that...,Terminology Manipulation
3,0,Armenian_Revolutionary_Federation_subsampled,The revision changes the description of the Ar...,Terminology Manipulation
4,0,Armenian_Revolutionary_Federation_subsampled,The added lines include significant historical...,Selective Insertion
...,...,...,...,...
2670,26,Armenia–Azerbaijan_border_crisis_(2021–present...,The revision reflects a significant shift in t...,Selective Omission
2671,26,Armenia–Azerbaijan_border_crisis_(2021–present...,The revision introduces a shift in terminology...,Selective Omission
2672,26,Armenia–Azerbaijan_border_crisis_(2021–present...,The revision introduces specific casualty figu...,Selective Omission
2673,26,Armenia–Azerbaijan_border_crisis_(2021–present...,The revision reflects a significant shift in t...,Selective Omission


## Cluster Naming Merge

In [12]:
import os
import pandas as pd
data_dir = ".."

# read cluster_topics_with_reduced_weaponization_techniques
cluster_topics_with_reduced_weaponization_techniques = pd.read_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_cluster_topics_with_reduced_weaponization_techniques.csv"))

# reorder columns to: cluster, topic, reduced_weaponization_technique, weaponization_technique, original_text, source
cluster_topics_with_reduced_weaponization_techniques = cluster_topics_with_reduced_weaponization_techniques[['cluster', 'topic', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]



In [13]:
# read existing names csv
names_path = os.path.join(data_dir, "keywords", "cluster_topics_named_text_directly.csv")
names_df = pd.read_csv(names_path)


# merge names_df with cluster_topics_with_reduced_weaponization_techniques on 'Cluster', "Topic" for the former and 'cluster', 'topic' for the latter

merged_df = pd.merge(cluster_topics_with_reduced_weaponization_techniques,
                     names_df[['Cluster', 'Topic', 'Cluster_Topic_Name']],
                     left_on=['cluster', 'topic'],
                     right_on=['Cluster', 'Topic'],
                     how='left')

# reorder so that the new cluster_name column is after cluster
cluster_topics_with_techniques_reduced = merged_df[['cluster', 'topic', 'Cluster_Topic_Name','reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

# save to csv
output_path = os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_cluster_topics_with_reduced_weaponization_techniques_named.csv")
cluster_topics_with_techniques_reduced.to_csv(output_path, index=False)

In [14]:
import os
import pandas as pd
data_dir = ".."

# read cluster_topics_with_reduced_weaponization_techniques
clusters_with_reduced_weaponization_techniques = pd.read_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_clusters_with_reduced_weaponization_techniques.csv"))

# reorder columns to: cluster, reduced_weaponization_technique, weaponization_technique, original_text, source

clusters_with_reduced_weaponization_techniques = clusters_with_reduced_weaponization_techniques[['cluster', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

clusters_with_reduced_weaponization_techniques.head()


,cluster,reduced_weaponization_technique,weaponization_technique,original_text,source
0,0,Linguistic Manipulation,Glorification & Vilification,"The revision introduces the term ""Armenian Rev...",Nagorno-Karabakh_conflict_subsampled
1,0,Linguistic Manipulation,Terminology Biasing,"The revision introduces the term ""Dashnaks"" in...",Armenian genocide_subsampled
2,0,Linguistic Manipulation,Terminology Biasing,The removed lines contain several phrases that...,Armenian_Revolutionary_Federation_subsampled
3,0,Linguistic Manipulation,Terminology Biasing,The revision changes the description of the Ar...,Armenian_Revolutionary_Federation_subsampled
4,0,Linguistic Manipulation,Glorification & Vilification,The added lines include significant historical...,Armenian_Revolutionary_Federation_subsampled


In [15]:
# read existing names csv
names_path = os.path.join(data_dir, "keywords", "cluster_keywords_named_text_directly.csv")
names_df = pd.read_csv(names_path)

# merge names_df with clusters_with_reduced_weaponization_techniques on 'Cluster' for the former and 'cluster' for the latter
merged_df = pd.merge(clusters_with_reduced_weaponization_techniques,
                     # drop columns Num_Comments,1-grams,2-grams,3-grams,
                     names_df.drop(columns=['Num_Comments', '1-grams', '2-grams', '3-grams']),
                     left_on='cluster',
                     right_on='Cluster',
                     how='left') 

# reorder so that the new cluster_name column is after cluster
merged_df = merged_df[['cluster', 'Cluster_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

# save merged_df to csv
merged_df.to_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_clusters_with_reduced_weaponization_techniques_named.csv"), index=False)

In [16]:
import os
import pandas as pd
data_dir = ".."

# read cluster_topics_with_reduced_weaponization_techniques
general_topics_with_reduced_weaponization_techniques = pd.read_csv(os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_general_topics_with_reduced_weaponization_techniques.csv"))

# reorder columns to: topic, reduced_weaponization_technique, weaponization_technique, original_text, source
general_topics_with_reduced_weaponization_techniques = general_topics_with_reduced_weaponization_techniques[['topic', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

In [17]:
# read existing names csv
keywords_path = os.path.join(data_dir, "keywords", "general_topics_named_text_directly.csv")
keywords_df = pd.read_csv(keywords_path)

# merge names_df with general_topics_with_reduced_weaponization_techniques on 'Topic' for the former and 'topic' for the latter
merged_df = pd.merge(general_topics_with_reduced_weaponization_techniques,
                     # drop columns Num_Comments,1-grams,2-grams,3-grams,
                     keywords_df.drop(columns=['Num_Documents', '1-grams', '2-grams', '3-grams']),
                     left_on='topic',
                     right_on='Topic',
                     how='left')  

# reorder so that the new topic_name column is after topic
general_topics_with_techniques_reduced = merged_df[['topic', 'Topic_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]   
# save to csv
output_path = os.path.join(data_dir, "other_outputs", "weaponization_analysis", "UPDATED_general_topics_with_reduced_weaponization_techniques_named.csv")
general_topics_with_techniques_reduced.to_csv(output_path, index=False)

#### Add Pro/Anti-Armenian Judgement

In [ ]:
# _finegrained_analysis.txt file format:

#Record 1 (Version: N/A):
#**Judgment:** Anti-Armenian

#**Explanation:** The removed line, "By the aid of France forces, some Armenian groups started to kill Turkic-muslim citizens of the city," framed Armenians as aggressors, which aligns with an anti-Armenian narrative. The addition of context regarding an "imminent Armenian insurrection" and the mob attack on the Armenian quarter shifts the focus to portraying Armenians as a threat. Phrases like "overheated Muslim population" suggest a justification for violence against Armenians, while the emphasis on "many thousand Armenians were killed" serves to evoke sympathy for the Muslim population. This selective presentation of historical events reinforces a narrative that vilifies Armenians and can be seen as an attempt to weaponize cultural heritage by framing the conflict in a divisive manner.
#--------------------------------------------------------------------------------
#Record 5 (Version: N/A):
#**Judgment:** Anti-Armenian

#**Explanation:** The added line references "Under the years of [[Assyrian Genocide|The Assyrian Genocide]]," which explicitly invokes the term "Assyrian Genocide." This term is politically charged and can be used to frame historical narratives that emphasize the victimhood of Assyrians while potentially downplaying or overshadowing the Armenian experience during similar historical events. The phrase "spirit of the Assyrian awareness" suggests a narrative of cultural resilience, which may serve to elevate the Assyrian perspective at the expense of the Armenian narrative. This addition could be interpreted as an attempt to shift focus away from the Armenian massacres and highlight the Assyrian plight, thus weaponizing the historical context to create a comparative narrative that may diminish the significance of the Armenian experience in the same period.
#--------------------------------------------------------------------------------
# etc etc

In [5]:
# Employ the best neighbors+components combination found above

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import word_tokenize, pos_tag

import os
import re
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from sklearn.metrics import silhouette_score
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
import pandas as pd
import spacy

np.random.seed(42)

nlp = spacy.load("en_core_web_md")

custom_stopwords = set([
    "armenia", "armenian", "armenians", "revision"
])


def parse_analysis_txt_list(analysis_path):
    with open(analysis_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    blocks = re.split(r'-{5,}', content)
    records = []
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        
        rec_match = re.search(r'Record\s+(\d+).*?\(Version:\s*([^)]+)\)', block, re.IGNORECASE)
        if rec_match:
            record_num = int(rec_match.group(1))
            version_val = rec_match.group(2).strip()
        else:
            record_num = None
            version_val = None
        
        judgment_match = re.search(r'\*\*Judgment:\*\*\s*(.*)', block, re.IGNORECASE)
        judgment_val = judgment_match.group(1).strip() if judgment_match else "Not Found"
        #print(judgment_val)
        analysis_match = re.search(r'\*\*Explanation:\*\*\s*(.*)', block, re.IGNORECASE)
        analysis_val = analysis_match.group(1).strip() if analysis_match else block
        
        records.append({
            "record": record_num,
            "version": version_val,
            "analysis": analysis_val,
            "judgment": judgment_val,
            "text": block
        })
    return records


def lemmatize_texts(texts):
    lemmatized = []
    all_texts = texts
    for doc in nlp.pipe(all_texts, batch_size=50):
        tokens = [token.lemma_.lower() for token in doc 
                  if not token.is_stop 
                  and not token.is_punct 
                  and token.lemma_.lower() not in custom_stopwords 
                  and len(token.lemma_) > 2]
        lemmatized.append(" ".join(tokens))
    return lemmatized

# ----------------------------
# Load comments+analysis text from all csv files
# ----------------------------

file_dir = "../csv_files"
for filename in os.listdir(file_dir):
    filename = os.path.join(file_dir, filename)
    if filename.endswith("_output.csv"):
        #print(filename)
        new_csv_rows = []

        analysis_file = filename.replace("/csv_files", "/txt_results_finegrained").replace("_output.csv", "_finegrained_analysis.txt")
        analysis_records = parse_analysis_txt_list(analysis_file)
        file_path = os.path.join(file_dir, filename)
        df = pd.read_csv(file_path, encoding="utf-8")
        df = df[df["Judgment"] == "Weaponised"]
        texts = df["Analysis"].dropna().astype(str).tolist()
        sourcename = filename.replace("_output.csv", "").replace("../csv_files/", "")

        for i in range(len(analysis_records)):
            analysis_data = analysis_records[i]
            finegrained_analysis = analysis_data.get("analysis", "")
            lemmatized_analysis = lemmatize_texts([finegrained_analysis])[0]
            finegrained_judgment = analysis_data.get("judgment", "Not Found")
            # finegrained_judgment here is either Pro-Armenian or Anti-Armenian ideally
            new_csv_rows.append({
                "source": sourcename,
                "finegrained_judgment": finegrained_judgment,
                "finegrained_analysis": finegrained_analysis,
                "lemmatized_analysis": lemmatized_analysis,
            })
        
        # Create and save DataFrame per file
        df = pd.DataFrame(new_csv_rows)
        output_csv = filename.replace("/csv_files", "/csv_files/finegrained_analysis").replace("_output.csv", "_finegrained_output.csv")
        df.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"Wrote {len(new_csv_rows)} rows to {output_csv}")



Wrote 174 rows to ../csv_files/finegrained_analysis/Armenia_subsampled_finegrained_output.csv
Wrote 80 rows to ../csv_files/finegrained_analysis/Yerevan_subsampled_finegrained_output.csv
Wrote 243 rows to ../csv_files/finegrained_analysis/Nagorno-Karabakh_subsampled_finegrained_output.csv
Wrote 64 rows to ../csv_files/finegrained_analysis/Adana_subsampled_finegrained_output.csv
Wrote 124 rows to ../csv_files/finegrained_analysis/Armenians_subsampled_finegrained_output.csv
Wrote 69 rows to ../csv_files/finegrained_analysis/Armenian_language_subsampled_finegrained_output.csv
Wrote 85 rows to ../csv_files/finegrained_analysis/Mount_Ararat_subsampled_finegrained_output.csv
Wrote 169 rows to ../csv_files/finegrained_analysis/Armenian_genocide_recognition_subsampled_finegrained_output.csv
Wrote 164 rows to ../csv_files/finegrained_analysis/Shusha_subsampled_finegrained_output.csv
Wrote 106 rows to ../csv_files/finegrained_analysis/Dolma_subsampled_finegrained_output.csv
Wrote 79 rows to ../c

## Finegrained factional merges

In [30]:
import os
import pandas as pd

data_dir = ".."

# same here, we generated technique categorization of clusters elsewhere first
clusters_with_techniques_path = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian_clusters_with_weaponization_techniques.csv")
cluster_topics_path = os.path.join(data_dir, "other_outputs", "finegrained", "pro-armenian_bertopic_per_cluster_topic_assignments.csv")
general_topics_path = os.path.join(data_dir, "other_outputs", "finegrained", "pro-armenian_bertopic_topic_assignments.csv")

clusters_with_techniques = pd.read_csv(clusters_with_techniques_path)
cluster_topics = pd.read_csv(cluster_topics_path)
general_topics = pd.read_csv(general_topics_path)

In [23]:
# merge clusters_with_techniques with general_topics on 'source', 'lemmatized_text', and 'original_text'
# BUT our final goal is to extract rows from general_topics that aren't in the merged dataframe
clusters_general_topics_merged = pd.merge(clusters_with_techniques.drop(columns=['cluster']),
                     general_topics.drop(columns=['lemmatized_text']),
                     on=['source', 'original_text'],
                     how='inner')
clusters_general_topics_merged.head()
# get rows from general_topics that aren't in clusters_general_topics_merged
general_topics_exclusive = general_topics.merge(clusters_general_topics_merged,
                                                on=['source', 'original_text', 'topic'],
                                                how='left',
                                                indicator=True)
general_topics_exclusive = general_topics_exclusive[general_topics_exclusive['_merge'] == 'left_only'].drop(columns=['_merge'])
general_topics_exclusive.head()

,topic,source,lemmatized_text,original_text,weaponization_technique
31,0,Armenian genocide_subsampled,alter title reference uzun yıl 1915 ermeni tec...,The revision alters the title of a reference f...,NaN
32,0,History_of_Armenia_subsampled,significantly alter estimate number death even...,The revision significantly alters the estimate...,NaN
41,0,Armenian genocide_subsampled,significantly reduce estimate fatality genocid...,The revision significantly reduces the estimat...,NaN
44,0,Armenian_Revolutionary_Federation_subsampled,reflect notable shift language align anti narr...,The revision reflects a notable shift in langu...,NaN
50,0,Armed_Forces_of_Armenia_subsampled,include addition highly offensive derogatory t...,The revision includes the addition of highly o...,NaN


In [ ]:
# save general_topics_exclusive to csv
general_topics_exclusive.to_csv(os.path.join(data_dir, "other_outputs", "finegrained", "pro-armenian_entries_exclusive_to_general_topics.csv"), index=False)

In [ ]:
# read general_topics_exclusive_with_technique aka the version AFTER the technique has been assigned

general_topics_exclusive_with_technique = pd.read_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian_topics-exclusive_with_weaponization_techniques.csv"))
general_topics_exclusive_with_technique.head()

,topic,source,original_text,weaponization_technique
0,0,Armenian genocide_subsampled,The revision alters the title of a reference f...,Terminology Biasing
1,0,History_of_Armenia_subsampled,The revision significantly alters the estimate...,Selective Omission
2,0,Armenian genocide_subsampled,The revision significantly reduces the estimat...,Selective Omission
3,0,Armenian_Revolutionary_Federation_subsampled,The revision reflects a notable shift in langu...,Terminology Biasing
4,0,Armed_Forces_of_Armenia_subsampled,The revision includes the addition of highly o...,Glorification & Vilification


In [ ]:
clusters_general_topics_merged = clusters_general_topics_merged[['topic', 'weaponization_technique', 'original_text', 'source']]

general_topics_exclusive_with_technique = general_topics_exclusive_with_technique[['topic', 'weaponization_technique', 'original_text', 'source']]

general_topics_with_technique = pd.concat([clusters_general_topics_merged, general_topics_exclusive_with_technique], ignore_index=True)

general_topics_with_technique.sort_values(by=['topic'], inplace=True)
# save to csv
general_topics_with_technique.to_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian-general_topics_with_weaponization_techniques.csv"), index=False)

In [27]:
general_topics_with_technique.head()

,topic,weaponization_technique,original_text,source
1550,0,Selective Omission,The removal of the entire passage regarding th...,Adana_subsampled
1549,0,Selective Omission,"The removal of the line ""* [http://www.tallarm...",Armenian genocide_subsampled
1548,0,Glorification & Vilification,"The removal of the phrase ""KATHERINE IS A HAIR...",Armenian genocide_subsampled
1547,0,Selective Omission,The revision alters the estimated fatalities o...,Armenian genocide_subsampled
1546,0,Glorification & Vilification,The revision includes the removal of a neutral...,Second Nagorno-Karabakh War_subsampled


In [31]:
# merge clusters_with_techniques with cluster_topics on 'source', 'cluster', 'lemmatized_text', and 'original_text'
# maybe inner join since we want only rows with a valid 'topic' assignment
cluster_topics_with_technique = pd.merge(clusters_with_techniques,
                     cluster_topics.drop(columns=['lemmatized_text']),
                     on=['source', 'cluster', 'original_text'],
                     how='inner')  
cluster_topics_with_technique.head()
# save cluster_topics_with_technique to csv
# new column order: cluster,topic,weaponization_technique,original_text,source
cluster_topics_with_technique = cluster_topics_with_technique[['cluster', 'topic', 'weaponization_technique', 'original_text', 'source']]
# sort by cluster and topic
cluster_topics_with_technique = cluster_topics_with_technique.sort_values(by=['cluster', 'topic'])
# save to csv
cluster_topics_with_technique.to_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian_cluster_topics_with_weaponization_techniques.csv"), index=False)

## Finegrained factional naming merges

In [12]:
stance = "anti-armenian"  # or "anti-armenian"

In [13]:
import os
import pandas as pd
data_dir = ".."

# read cluster_topics_with_reduced_weaponization_techniques
cluster_topics_with_reduced_weaponization_techniques = pd.read_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", f"{stance}_cluster_topics_with_reduced_weaponization_techniques.csv"))

# reorder columns to: cluster, topic, reduced_weaponization_technique, weaponization_technique, original_text, source
cluster_topics_with_reduced_weaponization_techniques = cluster_topics_with_reduced_weaponization_techniques[['cluster', 'topic', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

In [14]:
# read existing names csv
names_path = os.path.join(data_dir, "keywords", "finegrained", f"{stance}_cluster_topics_named_text_directly.csv")
names_df = pd.read_csv(names_path)


# merge names_df with cluster_topics_with_reduced_weaponization_techniques on 'Cluster', "Topic" for the former and 'cluster', 'topic' for the latter

merged_df = pd.merge(cluster_topics_with_reduced_weaponization_techniques,
                     names_df[['Cluster', 'Topic', 'Cluster_Topic_Name']],
                     left_on=['cluster', 'topic'],
                     right_on=['Cluster', 'Topic'],
                     how='left')

# reorder so that the new cluster_name column is after cluster
cluster_topics_with_techniques_reduced = merged_df[['cluster', 'topic', 'Cluster_Topic_Name','reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

# save to csv
output_path = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", f"{stance}_cluster_topics_with_reduced_weaponization_techniques_named.csv")
cluster_topics_with_techniques_reduced.to_csv(output_path, index=False)

In [15]:
import os
import pandas as pd
data_dir = ".."

# read cluster_topics_with_reduced_weaponization_techniques
clusters_with_reduced_weaponization_techniques = pd.read_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", f"{stance}_clusters_with_reduced_weaponization_techniques.csv"))

# reorder columns to: cluster, reduced_weaponization_technique, weaponization_technique, original_text, source

clusters_with_reduced_weaponization_techniques = clusters_with_reduced_weaponization_techniques[['cluster', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

clusters_with_reduced_weaponization_techniques.head()

,cluster,reduced_weaponization_technique,weaponization_technique,original_text,source
0,0,Linguistic Manipulation,Terminology Biasing,The revision alters the description of who dep...,Mount_Ararat_subsampled
1,0,Linguistic Manipulation,Terminology Biasing,"The revision introduces the word ""allegedly"" i...",History_of_Armenia_subsampled
2,0,Linguistic Manipulation,Terminology Biasing,"The revision replaces ""Mount Ararat,"" a term t...",Yerevan_subsampled
3,0,Linguistic Manipulation,Terminology Biasing,"The revision adds the phrase ""in [[Turkey]]"" t...",Yerevan_subsampled
4,0,Linguistic Manipulation,Terminology Biasing,"The revision adds the phrase ""in [[Turkey]]"" t...",Yerevan_subsampled


In [16]:
# read existing names csv
names_path = os.path.join(data_dir, "keywords", "finegrained", f"{stance}_cluster_keywords_named_text_directly.csv")
names_df = pd.read_csv(names_path)

# merge names_df with clusters_with_reduced_weaponization_techniques on 'Cluster' for the former and 'cluster' for the latter
merged_df = pd.merge(clusters_with_reduced_weaponization_techniques,
                     # drop columns Num_Comments,1-grams,2-grams,3-grams,
                     names_df.drop(columns=['Num_Comments', '1-grams', '2-grams', '3-grams']),
                     left_on='cluster',
                     right_on='Cluster',
                     how='left') 

# reorder so that the new cluster_name column is after cluster
merged_df = merged_df[['cluster', 'Cluster_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

# save merged_df to csv
merged_df.to_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", f"{stance}_clusters_with_reduced_weaponization_techniques_named.csv"), index=False)

In [17]:
import pandas as pd
data_dir = ".."

# read cluster_topics_with_reduced_weaponization_techniques
general_topics_with_reduced_weaponization_techniques = pd.read_csv(os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", f"{stance}_general_topics_with_reduced_weaponization_techniques.csv"))

# reorder columns to: topic, reduced_weaponization_technique, weaponization_technique, original_text, source
general_topics_with_reduced_weaponization_techniques = general_topics_with_reduced_weaponization_techniques[['topic', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]

In [ ]:
# read existing names csv
keywords_path = os.path.join(data_dir, "keywords", "finegrained", f"{stance}_general_topics_named_text_directly.csv")
keywords_df = pd.read_csv(keywords_path)

# merge names_df with general_topics_with_reduced_weaponization_techniques on 'Topic' for the former and 'topic' for the latter
merged_df = pd.merge(general_topics_with_reduced_weaponization_techniques,
                     # drop columns Num_Comments,1-grams,2-grams,3-grams,
                     keywords_df.drop(columns=['Num_Documents', '1-grams', '2-grams', '3-grams']),
                     left_on='topic',
                     right_on='Topic',
                     how='left')

# reorder so that the new topic_name column is after topic
general_topics_with_techniques_reduced = merged_df[['topic', 'Topic_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'source']]   
# save to csv
output_path = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", f"{stance}_general_topics_with_reduced_weaponization_techniques_named.csv")
general_topics_with_techniques_reduced.to_csv(output_path, index=False)

In [19]:
import os
import pandas as pd
data_dir = ".."

path_1 = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian_general_topics_with_reduced_weaponization_techniques_named.csv")
path_2 = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "anti-armenian_general_topics_with_reduced_weaponization_techniques_named.csv")
# concatenate the two dataframes
df1 = pd.read_csv(path_1)
df2 = pd.read_csv(path_2)
df1.drop(columns=['topic'], inplace=True)
df2.drop(columns=['topic'], inplace=True)
df1["judgment"] = "Pro-Armenian"
df2["judgment"] = "Anti-Armenian"
# put Judgement as the first column for both dataframes
df1 = df1[['judgment'] + [col for col in df1.columns if col != 'judgment']]
df2 = df2[['judgment'] + [col for col in df2.columns if col != 'judgment']]
merged_df = pd.concat([df1, df2], ignore_index=True)
# save to csv
output_path = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "all_general_topics_with_reduced_weaponization_techniques_named.csv")
merged_df.to_csv(output_path, index=False)

In [18]:
path_1 = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian_clusters_with_reduced_weaponization_techniques_named.csv")
path_2 = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "anti-armenian_clusters_with_reduced_weaponization_techniques_named.csv")
# concatenate the two dataframes
df1 = pd.read_csv(path_1)
df2 = pd.read_csv(path_2)

df1["judgment"] = "Pro-Armenian"
df2["judgment"] = "Anti-Armenian"
df1.drop(columns=['cluster'], inplace=True)
df2.drop(columns=['cluster'], inplace=True)
# put Judgement as the first column for both dataframes
df1 = df1[['judgment'] + [col for col in df1.columns if col != 'judgment']]
df2 = df2[['judgment'] + [col for col in df2.columns if col != 'judgment']]
merged_df = pd.concat([df1, df2], ignore_index=True)
# save to csv
output_path = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "all_clusters_with_reduced_weaponization_techniques_named.csv")
merged_df.to_csv(output_path, index=False)

In [17]:
path_1 = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "pro-armenian_cluster_topics_with_reduced_weaponization_techniques_named.csv")
path_2 = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "anti-armenian_cluster_topics_with_reduced_weaponization_techniques_named.csv")
# concatenate the two dataframes
df1 = pd.read_csv(path_1)
df2 = pd.read_csv(path_2)
df1["judgment"] = "Pro-Armenian"
df2["judgment"] = "Anti-Armenian"
df1.drop(columns=['cluster', 'topic'], inplace=True)
df2.drop(columns=['cluster', 'topic'], inplace=True)
# put Judgement as the first column for both dataframes
df1 = df1[['judgment'] + [col for col in df1.columns if col != 'judgment']]
df2 = df2[['judgment'] + [col for col in df2.columns if col != 'judgment']]
merged_df = pd.concat([df1, df2], ignore_index=True)
# save to csv
output_path = os.path.join(data_dir, "other_outputs", "finegrained", "weaponization_analysis", "all_cluster_topics_with_reduced_weaponization_techniques_named.csv")
merged_df.to_csv(output_path, index=False)

# Merge the OG Diff to the analysis files as well

In [20]:
import os
import json
from tqdm import tqdm
import numpy as np
import pandas as pd

np.random.seed(42)


# ----------------------------
# Load comments+analysis text from all csv files
# ----------------------------sx
file_dir = "../csv_files/finegrained_analysis"
file_dir_2 = "../csv_files"

total_df_pro = pd.DataFrame()

for filename in os.listdir(file_dir):
    # concat all csv files within the pro-armenian alignment
    if filename.endswith("_output.csv"):
        filename_2 = filename.replace("_finegrained", "")
        file_path = os.path.join(file_dir, filename)
        file_path_2 = os.path.join(file_dir_2, filename_2)
        df = pd.read_csv(file_path, encoding="utf-8")
        df_2 = pd.read_csv(file_path_2, encoding="utf-8")
        df["Timestamp"] = df_2["Timestamp"]
        df["User"] = df_2["User"]
        df["Diff"] = df_2["Diff"]
        # rearrange the colums to be User, Timestamp, finegrained_judgment, finegrained_analysis, weaponization_technique, reduced_weaponization_technique
        df = df[["User", "Timestamp", "source", "finegrained_judgment", "finegrained_analysis", "Diff"]]
        total_df_pro = pd.concat([total_df_pro, df], ignore_index=True)

#save_dir = "../other_outputs/finegrained/anti_armenian_total.csv"


In [37]:

df3 = pd.read_csv("../other_outputs/finegrained/weaponization_analysis/all_cluster_topics_with_reduced_weaponization_techniques_named.csv", encoding="utf-8")
# merge total_df_pro ith df3 where total_df_pro.finegrained_analysis == df3.original_text and for total_df_pro only User,	Timestamp, and Diff columns are kept

total_df_pro_final = pd.merge(total_df_pro[['User','Diff', 'source', 'finegrained_analysis']],
                        df3[['judgment', 'original_text', 'reduced_weaponization_technique', 'weaponization_technique', 'Cluster_Topic_Name']],
                        left_on='finegrained_analysis',
                        right_on='original_text',
                        how='right')

total_df_pro_final = total_df_pro_final[['judgment', 'Cluster_Topic_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'Diff', 'source']]


In [45]:
len(df3)

2344

In [38]:
len(total_df_pro_final)

2344

In [36]:
total_df_pro_final.to_csv("../other_outputs/finegrained/weaponization_analysis/WITH_DIFF_all_cluster_topics_with_reduced_weaponization_techniques_named.csv", index=False, encoding="utf-8")

In [39]:

df4 = pd.read_csv("../other_outputs/finegrained/weaponization_analysis/all_clusters_with_reduced_weaponization_techniques_named.csv", encoding="utf-8")
# merge total_df_pro ith df4 where total_df_pro.finegrained_analysis == df3.original_text and for total_df_pro only User,	Timestamp, and Diff columns are kept

total_df_pro_final = pd.merge(total_df_pro[['User','Diff', 'source', 'finegrained_analysis']],
                        df4[['judgment', 'original_text', 'reduced_weaponization_technique', 'weaponization_technique', 'Cluster_Name']],
                        left_on='finegrained_analysis',
                        right_on='original_text',
                        how='right')

total_df_pro_final = total_df_pro_final[['judgment', 'Cluster_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'Diff', 'source']]

In [44]:
len(df4)

3085

In [41]:
len(total_df_pro_final)

3085

In [25]:
total_df_pro_final.to_csv("../other_outputs/finegrained/weaponization_analysis/WITH_DIFF_all_clusters_with_reduced_weaponization_techniques_named.csv", index=False, encoding="utf-8")

In [42]:

df5 = pd.read_csv("../other_outputs/finegrained/weaponization_analysis/all_general_topics_with_reduced_weaponization_techniques_named.csv", encoding="utf-8")
# merge total_df_pro ith df5 where total_df_pro.finegrained_analysis == df3.original_text and for total_df_pro only User,	Timestamp, and Diff columns are kept

total_df_pro_final = pd.merge(total_df_pro[['User','Diff', 'source', 'finegrained_analysis']],
                        df5[['judgment', 'original_text', 'reduced_weaponization_technique', 'weaponization_technique', 'Topic_Name']],
                        left_on='finegrained_analysis',
                        right_on='original_text',
                        how='right')

total_df_pro_final = total_df_pro_final[['judgment', 'Topic_Name', 'reduced_weaponization_technique', 'weaponization_technique', 'original_text', 'Diff', 'source']]

In [46]:
len(df5)

3336

In [43]:
len(total_df_pro_final)

3336

In [27]:
total_df_pro_final.to_csv("../other_outputs/finegrained/weaponization_analysis/WITH_DIFF_all_general_topics_with_reduced_weaponization_techniques_named.csv", index=False, encoding="utf-8")

In [2]:
import os
import json
from tqdm import tqdm
import numpy as np
import pandas as pd

np.random.seed(42)


# ----------------------------
# Load comments+analysis text from all csv files
# ----------------------------sx
file_dir = "../csv_files/finegrained_analysis"
file_dir_2 = "../csv_files"

total_df_pro = pd.DataFrame()

for filename in os.listdir(file_dir):
    # concat all csv files within the pro-armenian alignment
    if filename.endswith("_output.csv"):
        filename_2 = filename.replace("_finegrained", "")
        file_path = os.path.join(file_dir, filename)
        file_path_2 = os.path.join(file_dir_2, filename_2)
        df = pd.read_csv(file_path, encoding="utf-8")
        df = df[df["finegrained_judgment"] == "Anti-Armenian"] # Filter to only "Anti-Armenian" judgments
        df_2 = pd.read_csv(file_path_2, encoding="utf-8")
        df["Timestamp"] = df_2["Timestamp"]
        df["User"] = df_2["User"]
        df["Diff"] = df_2["Diff"]
        # rearrange the colums to be User, Timestamp, finegrained_judgment, finegrained_analysis, weaponization_technique, reduced_weaponization_technique
        df = df[["User", "Timestamp", "source", "finegrained_judgment", "finegrained_analysis", "Diff"]]
        total_df_pro = pd.concat([total_df_pro, df], ignore_index=True)

save_dir = "../other_outputs/finegrained/anti_armenian_total.csv"

total_df_pro.to_csv(save_dir, index=False, encoding="utf-8")

In [ ]:

# read UPDATED_general_topics_with_reduced_weaponization_techniques_named